# Kaggle: MobileNetV4ConvSmall FSD50K Pretraining

This notebook trains a MobileNetV4 Conv Small audio classifier on FSD50K precomputed 1-second log-mel patches (`101` time frames x `96` mel bins) on Kaggle.

**Kaggle setup:**
1. Attach the `npy-fsd50k-patches-101tf-96mel` dataset to the notebook.
2. Select GPU acceleration with 2x NVIDIA T4 GPUs.
3. Run all cells in order.

**Current training changes:**
- Uses `tf.distribute.MirroredStrategy` so both T4 GPUs receive slices of each global batch.
- Keeps mixed precision and XLA enabled for T4 throughput.
- Monitors validation macro AUPRC for checkpointing, early stopping, and LR reduction.
- Calibrates decision thresholds on the validation split before reporting test F1.


## Step 1: Configure Kaggle Paths


In [ ]:
from pathlib import Path
import sys


ML_ROOT = Path('/kaggle/working')

# Ground-truth CSV + vocabulary location (labels/splits)
FSD50K_GROUND_TRUTH_DIR = Path('/kaggle/input/datasets/oselumeseagbonrofo/npy-fsd50k-patches-101tf-96mel/FSD50K.ground_truth/FSD50K.ground_truth')

# Dev patches path
FSD50K_FEATURE_DIR = Path('/kaggle/input/datasets/oselumeseagbonrofo/npy-fsd50k-patches-101tf-96mel/npy_fsd50k_dev_patches/npy_fsd50k_dev_patches')

# Eval patches path
EVAL_NPY_FSD50K_DIR = Path('/kaggle/input/datasets/oselumeseagbonrofo/npy-fsd50k-patches-101tf-96mel/npy_fsd50k_eval_patches/npy_fsd50k_eval_patches')

# Output directory
MODEL_OUTPUT_DIR = ML_ROOT / 'export' / 'mobilenetv4_convsmall_11'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_npy_files = list(FSD50K_FEATURE_DIR.glob('*.npy'))
eval_npy_files = list(EVAL_NPY_FSD50K_DIR.glob('*.npy'))

print(f'Found {len(train_npy_files)} train/validation .npy feature files')
print(f'Found {len(eval_npy_files)} eval .npy feature files')
print({
    'dataset_root': str(DATASET_ROOT),
    'ground_truth_dir': str(FSD50K_GROUND_TRUTH_DIR),
    'train_feature_dir': str(FSD50K_FEATURE_DIR),
    'eval_feature_dir': str(EVAL_NPY_FSD50K_DIR),
    'model_output_dir': str(MODEL_OUTPUT_DIR),
})


## Step 2: Install Dependencies

In [ ]:
%pip install -q tf-models-official tf-keras scikit-learn pandas numpy seaborn
print('Dependencies installed!')

In [ ]:
# Check GPU availability before creating the distribution strategy.
import tensorflow as tf

physical_gpus = tf.config.list_physical_devices('GPU')
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print(f'Could not set memory growth for {gpu.name}: {exc}')

print(f'TensorFlow version: {tf.__version__}')
print(f'Physical GPUs: {physical_gpus}')
if len(physical_gpus) < 2:
    print('WARNING: Fewer than 2 GPUs detected. On Kaggle, select the 2x T4 accelerator for the intended run.')


## Step 3: Setup and Configuration

In [ ]:
from __future__ import annotations

import json
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras as keras
from official.vision.modeling.backbones import mobilenet
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    hamming_loss,
    label_ranking_average_precision_score,
    multilabel_confusion_matrix,
    precision_recall_curve,
)

# Runtime acceleration toggles
SEED = 42
USE_MIXED_PRECISION = True
ENABLE_XLA = True
STEPS_PER_EXECUTION = 128

try:
    keras.utils.set_random_seed(SEED)
except AttributeError:
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

physical_gpus = tf.config.list_physical_devices('GPU')
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        # The runtime may already be initialized if the GPU check cell ran.
        pass

if ENABLE_XLA:
    tf.config.optimizer.set_jit(True)
if USE_MIXED_PRECISION:
    keras.mixed_precision.set_global_policy('mixed_float16')

DISTRIBUTION_STRATEGY = tf.distribute.MirroredStrategy()
NUM_REPLICAS = DISTRIBUTION_STRATEGY.num_replicas_in_sync

# Training configuration
MODEL_ID = 'MobileNetV4ConvSmall'
SAMPLE_RATE = 16000
WINDOW_DURATION_MS = 960
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_DURATION_MS / 1000)

# Only the mel feature contract matters here; the notebook does not re-run preprocessing.
PREPROCESSING_CONFIG = {
    'stft': {'n_fft': 1024, 'hop_length': 512, 'win_length': 1024},
    'mel': {'n_mels': 96, 'power': 2.0},
}
EXPECTED_MEL_BINS = PREPROCESSING_CONFIG['mel']['n_mels']
TIME_FRAMES = 101
IMAGE_SIZE = (EXPECTED_MEL_BINS, TIME_FRAMES)
EMBEDDING_DIM = 512
PER_REPLICA_BATCH_SIZE = 32
BATCH_SIZE = PER_REPLICA_BATCH_SIZE * NUM_REPLICAS
EPOCHS = 40
BASE_LEARNING_RATE = 3e-4
LEARNING_RATE = BASE_LEARNING_RATE * max(1, NUM_REPLICAS)
WEIGHT_DECAY = 1e-5
PREDICTION_THRESHOLD = 0.5
DROPOUT_RATE = 0.30
STOCHASTIC_DEPTH_DROP_RATE = 0.1
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.50
MAX_POSITIVE_CLASS_WEIGHT = 6.0
SHUFFLE_BUFFER_SIZE = 65536
AUTOTUNE = tf.data.AUTOTUNE

# Preprocessing summary for quick sanity checks in the notebook.
print('Configuration loaded')
print({
    'model_id': MODEL_ID,
    'sample_rate': SAMPLE_RATE,
    'preprocessing_config': PREPROCESSING_CONFIG,
    'image_size': IMAGE_SIZE,
    'num_replicas': NUM_REPLICAS,
    'per_replica_batch_size': PER_REPLICA_BATCH_SIZE,
    'global_batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'mixed_precision_policy': keras.mixed_precision.global_policy().name,
    'xla_enabled': ENABLE_XLA,
    'steps_per_execution': STEPS_PER_EXECUTION,
    'model_output_dir': str(MODEL_OUTPUT_DIR),
})


In [4]:
import numpy as np
from pathlib import Path

sample_feature_dir = FSD50K_FEATURE_DIR
expected_mels = PREPROCESSING_CONFIG['mel']['n_mels']

for feature_file in list(sample_feature_dir.glob('*.npy'))[:5]:
    feature = np.load(feature_file)
    print(feature_file.name, feature.shape, feature.dtype, feature.min(), feature.max(), np.mean(feature))
    if feature.ndim == 2 and expected_mels not in feature.shape:
        raise ValueError(
            f'{feature_file.name} does not match the expected mel-bin contract of {expected_mels} bins.'
        )


255902_032000.npy (101, 96) float16 -80.0 0.0 -55.78
145899_000000.npy (101, 96) float16 -80.0 0.0 -65.56
326392_088000.npy (101, 96) float16 -54.34 0.0 -25.72
95959_160000.npy (101, 96) float16 -80.0 0.0 -44.62
75435_024000.npy (101, 96) float16 -80.0 0.0 -29.9


## Step 4: Load FSD50K Dataset

In [5]:
def load_fsd50k_vocabulary(vocabulary_path: Path) -> pd.DataFrame:
    vocabulary = pd.read_csv(vocabulary_path, header=None, names=['index', 'label', 'mid'])
    vocabulary = vocabulary.sort_values('index').reset_index(drop=True)
    assert vocabulary['index'].tolist() == list(range(len(vocabulary))), 'FSD50K vocabulary indices must be contiguous.'
    return vocabulary


def labels_to_multihot(labels: str) -> np.ndarray:
    target = np.zeros(len(LABEL_ORDER), dtype=np.float32)
    for label in labels.split(','):
        index = LABEL_TO_INDEX.get(label)
        if index is not None:
            target[index] = 1.0
    return target


def load_fsd50k_split_patches(csv_path: Path, data_root: Path, split_overrides: dict[str | None, str]) -> pd.DataFrame:
    """Load FSD50K split and map each patch to its clip's labels."""
    frame = pd.read_csv(csv_path)
    if 'split' not in frame.columns:
        frame['split'] = None
    frame['split_name'] = frame['split'].map(split_overrides)
    frame = frame[frame['split_name'].notna()].copy()

    # Ensure fname is a string
    frame['fname_str'] = frame['fname'].astype(str)

    # Get all patch files from disk
    patch_files = sorted(data_root.glob('*.npy'))

    # Parse patch files: extract clip ID (everything before last underscore)
    patch_map = {}  # clip_id -> list of patch paths
    for patch_file in patch_files:
        # Filename format: {clip_id}_{offset:06d}.npy
        parts = patch_file.stem.rsplit('_', 1)
        if len(parts) == 2:
            clip_id = parts[0]
            if clip_id not in patch_map:
                patch_map[clip_id] = []
            patch_map[clip_id].append(str(patch_file))

    print(f'\n--- Debugging load_fsd50k_split_patches for {csv_path.name} ---')
    print(f'Found {len(patch_map)} unique clips with patches')
    print(f'Clip IDs from CSV: {frame["fname_str"].nunique()}')

    # Expand frame: each patch gets its own row with the clip's label
    expanded_rows = []
    for _, row in frame.iterrows():
        clip_id = row['fname_str']
        if clip_id in patch_map:
            for patch_path in patch_map[clip_id]:
                expanded_rows.append({
                    'fname': row['fname'],
                    'labels': row['labels'],
                    'mids': row['mids'],
                    'split_name': row['split_name'],
                    'data_path': patch_path,
                    'target': labels_to_multihot(row['labels'])
                })

    frame = pd.DataFrame(expanded_rows)

    if len(frame) == 0:
        print(f'WARNING: No patches found for {csv_path.name}')
    else:
        print(f'Expanded to {len(frame)} patch examples from {len(expanded_rows)} clips')

    print('--- End Debugging ---\n')
    return frame[['fname', 'labels', 'mids', 'split_name', 'data_path', 'target']]


vocabulary_df = load_fsd50k_vocabulary(FSD50K_GROUND_TRUTH_DIR / 'vocabulary.csv')
LABEL_ORDER = vocabulary_df['label'].tolist()
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABEL_ORDER)}

print(f'Loaded {len(LABEL_ORDER)} audio classes')

Loaded 200 audio classes


In [ ]:
# Load train/validation/test splits with patches.
train_validation_df = load_fsd50k_split_patches(
    FSD50K_GROUND_TRUTH_DIR / 'dev.csv',
    FSD50K_FEATURE_DIR,
    split_overrides={'train': 'train', 'val': 'validation'},
)

test_df = load_fsd50k_split_patches(
    FSD50K_GROUND_TRUTH_DIR / 'eval.csv',
    EVAL_NPY_FSD50K_DIR,
    split_overrides={None: 'test'},
) if EVAL_NPY_FSD50K_DIR.exists() else pd.DataFrame(columns=train_validation_df.columns)

fsd50k_df = pd.concat([train_validation_df, test_df], ignore_index=True)
assert not fsd50k_df.empty, 'No FSD50K patch examples were found.'
assert fsd50k_df['target'].map(lambda target: target.sum() > 0).all(), 'Every patch must have at least one label.'

split_summary = fsd50k_df.groupby('split_name').size().reindex(['train', 'validation', 'test'])
all_targets = np.stack(fsd50k_df['target'].to_numpy()).astype(np.float32)
label_frequency = all_targets.sum(axis=0)
train_targets_for_weights = np.stack(fsd50k_df.loc[fsd50k_df['split_name'] == 'train', 'target'].to_numpy()).astype(np.float32)
train_label_frequency = train_targets_for_weights.sum(axis=0)
train_negative_frequency = train_targets_for_weights.shape[0] - train_label_frequency

# The latest run had far more false negatives than false positives. These clipped
# sqrt inverse-frequency weights increase positive loss for rare labels without
# letting the rarest labels dominate the batch loss.
raw_positive_weights = np.sqrt((train_negative_frequency + 1.0) / (train_label_frequency + 1.0))
CLASS_POSITIVE_WEIGHTS = np.where(train_label_frequency > 0, raw_positive_weights, 1.0)
CLASS_POSITIVE_WEIGHTS = np.clip(CLASS_POSITIVE_WEIGHTS, 1.0, MAX_POSITIVE_CLASS_WEIGHT).astype(np.float32)

print({
    'num_classes': len(LABEL_ORDER),
    'split_counts': split_summary.fillna(0).astype(int).to_dict(),
    'available_patches': int(len(fsd50k_df)),
    'nonzero_labels': int(np.count_nonzero(label_frequency)),
    'positive_weight_min': float(CLASS_POSITIVE_WEIGHTS.min()),
    'positive_weight_median': float(np.median(CLASS_POSITIVE_WEIGHTS)),
    'positive_weight_max': float(CLASS_POSITIVE_WEIGHTS.max()),
})
if test_df.empty:
    print('WARNING: No eval patches detected. Training/validation will run, but test evaluation will be skipped.')

print('\nSplit summary:')
print(split_summary)

rare_label_summary = pd.DataFrame({
    'label': LABEL_ORDER,
    'train_positive_count': train_label_frequency.astype(int),
    'positive_weight': CLASS_POSITIVE_WEIGHTS,
}).sort_values('train_positive_count', ascending=True)
print('\nRarest training labels:')
print(rare_label_summary.head(10))

del all_targets


## Step 5: Audio Processing Functions

In [ ]:
def load_feature_array(feature_path: tf.Tensor) -> tf.Tensor:
    path = feature_path.numpy().decode('utf-8')
    feature = np.load(path)
    feature = np.asarray(feature, dtype=np.float32)

    # Normalise the array shape so downstream layers always see [mel, time].
    if feature.ndim == 1:
        feature = np.expand_dims(feature, axis=0)
    if feature.ndim != 2:
        feature = np.squeeze(feature)
        if feature.ndim != 2:
            raise ValueError(f'Unexpected feature shape: {feature.shape} for {path}')

    expected_mels = PREPROCESSING_CONFIG['mel']['n_mels']
    if feature.shape[0] == expected_mels:
        return feature
    if feature.shape[1] == expected_mels:
        return feature.T

    raise ValueError(
        f'Unexpected mel shape {feature.shape} for {path}; expected one axis to be {expected_mels} bins.'
    )


def mel_feature_to_image(feature: tf.Tensor) -> tf.Tensor:
    feature = tf.convert_to_tensor(feature, dtype=tf.float32)
    feature = tf.ensure_shape(feature, [IMAGE_SIZE[0], IMAGE_SIZE[1]])
    feature = tf.expand_dims(feature, axis=-1)
    feature = tf.image.per_image_standardization(feature)
    feature.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 1])
    return feature


def random_time_shift_tf(feature: tf.Tensor, max_shift: int = 8) -> tf.Tensor:
    shift = tf.random.uniform([], -max_shift, max_shift + 1, dtype=tf.int32)
    return tf.roll(feature, shift=shift, axis=1)


# SpecAugment implementation for [mel, time, channel] tensors.
def spec_augment_tf(
    feature: tf.Tensor,
    num_mask: int = 2,
    freq_masking_max_percentage: float = 0.15,
    time_masking_max_percentage: float = 0.30,
) -> tf.Tensor:
    feature = tf.identity(feature)
    shape = tf.shape(feature)

    freq_bins = shape[0]
    time_steps = shape[1]
    channels = shape[2]

    for _ in range(num_mask):
        freq_pct = tf.random.uniform([], 0, freq_masking_max_percentage)
        freq_width = tf.cast(tf.round(freq_pct * tf.cast(freq_bins, tf.float32)), tf.int32)
        freq_width = tf.maximum(freq_width, 0)
        freq_start = tf.cond(
            freq_bins - freq_width > 0,
            lambda: tf.random.uniform([], 0, freq_bins - freq_width + 1, dtype=tf.int32),
            lambda: tf.constant(0, dtype=tf.int32),
        )

        time_pct = tf.random.uniform([], 0, time_masking_max_percentage)
        time_width = tf.cast(tf.round(time_pct * tf.cast(time_steps, tf.float32)), tf.int32)
        time_width = tf.maximum(time_width, 0)
        time_start = tf.cond(
            time_steps - time_width > 0,
            lambda: tf.random.uniform([], 0, time_steps - time_width + 1, dtype=tf.int32),
            lambda: tf.constant(0, dtype=tf.int32),
        )

        # Frequency mask along the mel axis.
        top = feature[:freq_start, :, :]
        middle = tf.zeros([freq_width, time_steps, channels], dtype=feature.dtype)
        bottom = feature[freq_start + freq_width:, :, :]
        feature = tf.concat([top, middle, bottom], axis=0)

        # Time mask along the frame axis.
        left = feature[:, :time_start, :]
        middle = tf.zeros([freq_bins, time_width, channels], dtype=feature.dtype)
        right = feature[:, time_start + time_width:, :]
        feature = tf.concat([left, middle, right], axis=1)

    feature.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 1])
    return feature


def load_example(data_path: tf.Tensor, target: tf.Tensor, training: bool):
    feature = tf.py_function(load_feature_array, [data_path], Tout=tf.float32)
    feature.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1]])
    image = mel_feature_to_image(feature)
    if training:
        image = random_time_shift_tf(image)
        image = spec_augment_tf(image)
        image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 1])
    target.set_shape([len(LABEL_ORDER)])
    return image, target

print('Numpy-feature processing functions defined')


## Step 6: Create Datasets

In [ ]:
def make_dataset(split_name: str, training: bool) -> tf.data.Dataset:
    frame = fsd50k_df[fsd50k_df['split_name'] == split_name].copy()
    assert not frame.empty, f'No examples available for split: {split_name}'
    targets = np.stack(frame['target'].to_numpy()).astype(np.float32)
    dataset = tf.data.Dataset.from_tensor_slices((frame['data_path'].to_numpy(), targets))

    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.OFF
    if hasattr(options, 'experimental_slack'):
        options.experimental_slack = True
    if training:
        options.experimental_deterministic = False
    dataset = dataset.with_options(options)

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(frame), SHUFFLE_BUFFER_SIZE),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        lambda data_path, target: load_example(data_path, target, training=training),
        num_parallel_calls=AUTOTUNE,
        deterministic=not training,
    )
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=training)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset


train_ds = make_dataset('train', training=True)
validation_ds = make_dataset('validation', training=False)
test_ds = make_dataset('test', training=False) if (fsd50k_df['split_name'] == 'test').any() else None

print('Datasets created')
print(f'Train dataset: {train_ds}')
print(f'Validation dataset: {validation_ds}')
print(f'Test dataset: {test_ds}')


## Step 7: Build Model

In [ ]:
def build_mobilenetv4_classifier(num_classes: int) -> tuple[keras.Model, keras.Model]:
    inputs = keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 1), name='log_mel')
    backbone = mobilenet.MobileNet(
        model_id=MODEL_ID,
        input_specs=keras.layers.InputSpec(shape=[None, IMAGE_SIZE[0], IMAGE_SIZE[1], 1]),
        stochastic_depth_drop_rate=STOCHASTIC_DEPTH_DROP_RATE,
        output_stride=32,
        name='mobilenetv4_backbone',
    )
    backbone.trainable = True
    endpoints = backbone(inputs)
    final_endpoint_key = sorted([key for key in endpoints if key.isdigit()], key=int)[-1]
    x = endpoints[final_endpoint_key]
    if len(x.shape) == 4:
        x = keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = keras.layers.Dropout(DROPOUT_RATE, name='pre_embedding_dropout')(x)
    embedding = keras.layers.Dense(EMBEDDING_DIM, name='embedding')(x)
    normalized_embedding = keras.layers.Lambda(
        lambda tensor: tf.math.l2_normalize(tf.cast(tensor, tf.float32), axis=-1),
        dtype='float32',
        name='embedding_norm',
    )(embedding)
    outputs = keras.layers.Dense(
        num_classes,
        activation='sigmoid',
        dtype='float32',
        name='fsd50k_multilabel_classifier',
    )(normalized_embedding)
    classifier = keras.Model(inputs=inputs, outputs=outputs, name='soundpulse_mobilenetv4_fsd50k_classifier')
    embedding_model = keras.Model(inputs=inputs, outputs=normalized_embedding, name='soundpulse_mobilenetv4_embedding_model')
    return classifier, embedding_model


with DISTRIBUTION_STRATEGY.scope():
    classifier, embedding_model = build_mobilenetv4_classifier(len(LABEL_ORDER))
    optimizer = keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    classifier.compile(
        optimizer=optimizer,
        loss=keras.losses.BinaryFocalCrossentropy(
            gamma=FOCAL_GAMMA,
            alpha=FOCAL_ALPHA,
        ),
        metrics=[
            keras.metrics.BinaryAccuracy(name='binary_accuracy', threshold=PREDICTION_THRESHOLD),
            keras.metrics.AUC(name='micro_auprc', curve='PR', multi_label=False, num_thresholds=200),
        ],
        jit_compile=ENABLE_XLA,
        steps_per_execution=STEPS_PER_EXECUTION,
    )

print('Model built and compiled')
print(f'Model ID: {MODEL_ID}')
print(f'Distribution replicas: {NUM_REPLICAS}')


## Step 8: Train Model

In [ ]:
checkpoint_path = MODEL_OUTPUT_DIR / 'best.weights.h5'

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='micro_auprc',
        mode='max',
        save_best_only=True,
        save_weights_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor='micro_auprc',
        mode='max',
        patience=6,
        min_delta=1e-4,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='micro_auprc',
        mode='max',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(MODEL_OUTPUT_DIR / 'training_history.csv')),
]

history = classifier.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

if checkpoint_path.exists():
    classifier.load_weights(str(checkpoint_path))

pd.DataFrame(history.history).to_json(MODEL_OUTPUT_DIR / 'training_history.json', orient='records', indent=2)
print('\nTraining completed')


## Step 9: Evaluate Model

### Calculate lwrap

In [13]:
import sklearn.metrics
# Calculate the overall lwlrap using sklearn.metrics function.

def calculate_overall_lwlrap_sklearn(truth, scores):
  """Calculate the overall lwlrap using sklearn.metrics.lrap."""
  # sklearn doesn't correctly apply weighting to samples with no labels, so just skip them.
  sample_weight = np.sum(truth > 0, axis=1)
  nonzero_weight_sample_indices = np.flatnonzero(sample_weight > 0)
  overall_lwlrap = sklearn.metrics.label_ranking_average_precision_score(
      truth[nonzero_weight_sample_indices, :] > 0,
      scores[nonzero_weight_sample_indices, :],
      sample_weight=sample_weight[nonzero_weight_sample_indices])
  return overall_lwlrap

In [ ]:
def compute_multilabel_metrics(
    targets: np.ndarray,
    probabilities: np.ndarray
) -> dict[str, object]:
    predictions = predictions_from_thresholds(probabilities, thresholds)
    mcm = multilabel_confusion_matrix(targets, predictions)
    return {
        'micro_f1': float(f1_score(targets, predictions, average='micro', zero_division=0)),
        'macro_f1': float(f1_score(targets, predictions, average='macro', zero_division=0)),
        'micro_average_precision': float(average_precision_score(targets, probabilities, average='micro')),
        'macro_average_precision': float(average_precision_score(targets, probabilities, average='macro')),
        'label_ranking_average_precision': float(label_ranking_average_precision_score(targets, probabilities)),
        'label_weighted_ranking_average_precision': float(calculate_overall_lwlrap_sklearn(targets, probabilities)),
        'hamming_loss': float(hamming_loss(targets, predictions)),
        'Multilabel Confusion Matrix': np.sum(mcm, axis=0).tolist(),
    }


def print_metric_block(title: str, metric_block: dict[str, object]) -> None:
    print(f'\n{title}')
    for metric, value in metric_block.items():
        if metric == 'Multilabel Confusion Matrix':
            print(f'{metric}:\n{np.array(value)}')
        elif isinstance(value, float):
            print(f'{metric}: {value:.4f}')


metrics = compute_multilabel_metrics(test_targets, test_probabilities)

with (MODEL_OUTPUT_DIR / 'fsd50k_test_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2)

print_metric_block('Test Metrics - validation-calibrated per-class thresholds', metrics)


In [15]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame for easier plotting
label_counts_df = pd.DataFrame({
    'label': LABEL_ORDER,
    'frequency': label_frequency
})

# Sort by frequency to easily see underrepresented classes
label_counts_df = label_counts_df.sort_values('frequency', ascending=True)

#View classes with the lowest labels
print(label_counts_df.head(20))

                           label  frequency
165                        Tabla      514.0
157                       Sneeze     1044.0
84                          Gasp     1062.0
55        Cupboard_open_or_close     1173.0
159           Speech_synthesizer     1242.0
149                         Sigh     1293.0
77               Finger_snapping     1357.0
191  Whoosh_and_swoosh_and_swish     1359.0
182                   Typewriter     1366.0
37           Chuckle_and_chortle     1389.0
45                       Cowbell     1404.0
87                  Glockenspiel     1414.0
166                   Tambourine     1517.0
1                      Accordion     1524.0
22        Burping_and_eructation     1631.0
113        Marimba_and_xylophone     1672.0
136             Ratchet_and_pawl     1688.0
13                  Bicycle_bell     1723.0
46                         Crack     1750.0
173                         Tick     1799.0


In [16]:
#View classes with the lowest labels

print(label_counts_df.tail(20))

                                             label  frequency
104                             Keyboard_(musical)    23616.0
15   Bird_vocalization_and_bird_call_and_bird_song    25434.0
108                                         Liquid    27094.0
195        Wind_instrument_and_woodwind_instrument    27440.0
158                                         Speech    28444.0
125                                     Percussion    29485.0
115                                     Mechanisms    29615.0
4                                            Alarm    31431.0
100                            Human_group_actions    33454.0
70                                          Engine    34540.0
14                                            Bird    38046.0
118                           Motor_vehicle_(road)    46607.0
187                                          Water    50493.0
192                                   Wild_animals    54934.0
101                                    Human_voice    71139.0
5       

## Step 10: Save Model Artifacts

In [ ]:
# Collect embeddings.
def collect_embeddings(dataset: tf.data.Dataset, split_name: str) -> tuple[np.ndarray, np.ndarray]:
    embeddings = embedding_model.predict(dataset, verbose=1)
    targets = np.stack(fsd50k_df.loc[fsd50k_df['split_name'] == split_name, 'target'].to_numpy()).astype(np.float32)
    if embeddings.shape[0] != targets.shape[0]:
        raise ValueError(
            f'Embedding/target length mismatch for {split_name}: '
            f'{embeddings.shape[0]} embeddings vs {targets.shape[0]} targets'
        )
    return embeddings.astype(np.float32), targets


if test_ds is None:
    print('Embedding export skipped: no test dataset available.')
else:
    test_embeddings, test_embedding_targets = collect_embeddings(test_ds, 'test')
    np.save(MODEL_OUTPUT_DIR / 'test_embeddings.npy', test_embeddings)
    np.save(MODEL_OUTPUT_DIR / 'test_targets.npy', test_embedding_targets)

    print(f'Test embeddings shape: {test_embeddings.shape}')


In [ ]:
# Save models.
classifier.save(str(MODEL_OUTPUT_DIR / 'classifier.keras'), include_optimizer=False)
embedding_model.save(str(MODEL_OUTPUT_DIR / 'embedding_model.keras'), include_optimizer=False)
embedding_model.export(str(MODEL_OUTPUT_DIR / 'saved_model'))

# Convert to TFLite.
converter = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_OUTPUT_DIR / 'saved_model'))
# Allow TensorFlow operations needed by the official MobileNetV4 backbone.
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
tflite_model = converter.convert()
with (MODEL_OUTPUT_DIR / 'mobilenetv4_conv_small_embedding.tflite').open('wb') as handle:
    handle.write(tflite_model)

class_thresholds = globals().get('CLASS_THRESHOLDS')
threshold_summary = globals().get('THRESHOLD_CALIBRATION_SUMMARY')

# Save metadata.
metadata = {
    'model_name': MODEL_ID,
    'training_mode': 'from_scratch',
    'dataset': 'FSD50K',
    'task': 'multi_label_audio_classification',
    'num_classes': len(LABEL_ORDER),
    'label_order': LABEL_ORDER,
    'embedding_dim': EMBEDDING_DIM,
    'image_size': IMAGE_SIZE,
    'sample_rate_hz': SAMPLE_RATE,
    'window_duration_ms': WINDOW_DURATION_MS,
    'preprocessing_config': PREPROCESSING_CONFIG,
    'num_replicas': NUM_REPLICAS,
    'per_replica_batch_size': PER_REPLICA_BATCH_SIZE,
    'global_batch_size': BATCH_SIZE,
    'mixed_precision': USE_MIXED_PRECISION,
    'xla_enabled': ENABLE_XLA,
    'base_learning_rate': BASE_LEARNING_RATE,
    'effective_learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'dropout_rate': DROPOUT_RATE,
    'stochastic_depth_drop_rate': STOCHASTIC_DEPTH_DROP_RATE,
    'focal_gamma': FOCAL_GAMMA,
    'focal_alpha': FOCAL_ALPHA,
    'max_positive_class_weight': MAX_POSITIVE_CLASS_WEIGHT,
    'threshold_strategy': 'validation_per_class_f1' if class_thresholds is not None else 'fixed_0_5',
    'prediction_threshold': float(globals().get('GLOBAL_F1_THRESHOLD', PREDICTION_THRESHOLD)),
    'class_thresholds': class_thresholds.tolist() if class_thresholds is not None else None,
    'threshold_calibration_summary': threshold_summary,
}
with (MODEL_OUTPUT_DIR / 'export_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'Models saved to {MODEL_OUTPUT_DIR}')
